# Strategy Recommendations & Policy Framework
## Notebook 5: Generate actionable policy recommendations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set_style('whitegrid')

# Load all processed data
lending_data = pd.read_csv('../data/processed/lending_data_segmented.csv')
segment_profile = pd.read_csv('../data/processed/segment_profile.csv', index_col=0)
segment_profit = pd.read_csv('../data/processed/segment_profitability.csv', index_col=0)
early_warnings = pd.read_csv('../data/processed/early_warning_signals.csv')

print('All data loaded successfully!')

## Step 1: Executive Summary Metrics

In [ ]:
# Calculate executive summary metrics
exec_summary = {
    'Total Active Loans': len(lending_data),
    'Total Disbursed Amount': lending_data['loan_amount'].sum(),
    'Portfolio Default Rate': lending_data['is_default'].mean(),
    'Portfolio at Risk (>30 DPD)': (lending_data['days_past_due'] > 30).mean(),
    'Average Interest Rate': lending_data['interest_rate'].mean(),
    'Total Expected Loss': (lending_data['loan_amount'] * lending_data['is_default'].astype(float) * 0.45).sum(),
    'Total Acquisition Cost': lending_data['acquisition_cost'].sum(),
    'Average Customer Credit Score': lending_data['credit_score_proxy'].mean(),
}

print('='*70)
print('EXECUTIVE SUMMARY - PORTFOLIO METRICS')
print('='*70)
for key, value in exec_summary.items():
    if 'Amount' in key or 'Loss' in key or 'Cost' in key:
        print(f'{key}: ₹{value:,.2f}')
    elif 'Rate' in key or 'Risk' in key:
        print(f'{key}: {value:.2%}')
    else:
        print(f'{key}: {value:.2f}')

## Step 2: Top Risk Segments & Recommendations

In [ ]:
# Identify worst performing segments by geography, channel, and product
geo_risk = lending_data.groupby('geography').agg({
    'is_default': 'mean',
    'loan_id': 'count',
    'days_past_due': 'mean'
}).sort_values('is_default', ascending=False)

channel_risk = lending_data.groupby('acquisition_channel').agg({
    'is_default': 'mean',
    'loan_id': 'count',
    'acquisition_cost': 'mean'
}).sort_values('is_default', ascending=False)

product_risk = lending_data.groupby('product_type').agg({
    'is_default': 'mean',
    'loan_id': 'count',
    'interest_rate': 'mean'
}).sort_values('is_default', ascending=False)

print('\n' + '='*70)
print('TOP RISK SEGMENTS')
print('='*70)
print('\nBY GEOGRAPHY:')
print(geo_risk.round(4))
print('\nBY ACQUISITION CHANNEL:')
print(channel_risk.round(4))
print('\nBY PRODUCT TYPE:')
print(product_risk.round(4))

## Step 3: Policy Recommendations - Pricing Adjustments

In [ ]:
# Recommend pricing adjustments by segment
# Formula: New Rate = Current Rate + (Default Rate Increase * Rate Elasticity)

pricing_recs = segment_profit.copy()
base_rate = lending_data['interest_rate'].mean()

# Calculate recommended rate based on risk
pricing_recs['Current_Avg_Rate'] = 0
pricing_recs['Recommended_Rate'] = 0
pricing_recs['Rate_Adjustment_Bps'] = 0

for tier in pricing_recs.index:
    tier_data = lending_data[lending_data['risk_tier'] == tier]
    current_rate = tier_data['interest_rate'].mean()
    default_rate = tier_data['is_default'].mean()
    
    # Risk premium = default rate * 500 bps (calibration factor)
    risk_premium = default_rate * 500
    recommended_rate = current_rate + (risk_premium / 100)
    
    pricing_recs.loc[tier, 'Current_Avg_Rate'] = current_rate
    pricing_recs.loc[tier, 'Recommended_Rate'] = min(recommended_rate, 36)  # Cap at 36%
    pricing_recs.loc[tier, 'Rate_Adjustment_Bps'] = (recommended_rate - current_rate) * 100

pricing_recs = pricing_recs[['Default_Rate', 'Loan_Count', 'Current_Avg_Rate', 'Recommended_Rate', 'Rate_Adjustment_Bps', 'Expected_Loss', 'Net_Income']]

print('\n' + '='*70)
print('PRICING POLICY RECOMMENDATIONS')
print('='*70)
print(pricing_recs.round(2))

## Step 4: Impact Analysis - Expected Loss Reduction

In [ ]:
# Calculate impact of policy recommendations
print('\n' + '='*70)
print('IMPACT ANALYSIS - POLICY RECOMMENDATIONS')
print('='*70)

# Current portfolio metrics
current_el = (lending_data['loan_amount'] * lending_data['is_default'].astype(float) * 0.45).sum()
current_net_income = (lending_data['loan_amount'] * (lending_data['interest_rate'] / 100) - (lending_data['loan_amount'] * lending_data['is_default'].astype(float) * 0.45)).sum()

# Projected metrics with pricing adjustments
# Assumption: 1% increase in rate reduces default rate by 0.5% (interest sensitivity)
projected_el = 0
projected_income = 0

for tier in pricing_recs.index:
    tier_data = lending_data[lending_data['risk_tier'] == tier]
    tier_size = tier_data['loan_amount'].sum()
    rate_adj_pct = pricing_recs.loc[tier, 'Rate_Adjustment_Bps'] / 100
    
    # Projected default rate reduction from rate increase
    default_reduction = (tier_data['is_default'].mean() * (rate_adj_pct * 0.05))
    projected_default_rate = max(tier_data['is_default'].mean() - default_reduction, 0.01)
    
    # Projected EL and income
    projected_el += tier_size * projected_default_rate * 0.45
    projected_income += tier_size * (pricing_recs.loc[tier, 'Recommended_Rate'] / 100)

el_reduction = current_el - projected_el
income_increase = projected_income - (lending_data['loan_amount'].sum() * (lending_data['interest_rate'].mean() / 100))

print(f'\nCurrent Expected Loss: ₹{current_el:,.2f}')
print(f'Projected Expected Loss (with new pricing): ₹{projected_el:,.2f}')
print(f'Expected Loss Reduction: ₹{el_reduction:,.2f} ({el_reduction/current_el:.1%})')

print(f'\nCurrent Annual Interest Income: ₹{lending_data["loan_amount"].sum() * (lending_data["interest_rate"].mean() / 100):,.2f}')
print(f'Projected Annual Interest Income: ₹{projected_income:,.2f}')
print(f'Additional Income from Repricing: ₹{income_increase:,.2f}')

net_benefit = el_reduction + income_increase
print(f'\nTotal Net Benefit (EL Reduction + Additional Income): ₹{net_benefit:,.2f}')

## Step 5: Early Warning Action Plan

In [ ]:
# Define actions for early warning levels
action_plan = {
    'No_Risk': 'Continue standard monitoring. Annual review.',
    'Low_Risk': 'Quarterly check-in. Monitor cash flow trends.',
    'Medium_Risk': 'Monthly contact. Discuss payment restructuring options. Consider rate adjustment.',
    'High_Risk': 'Weekly contact. Offer early intervention (tenure extension, payment holiday). Flag for collections readiness.',
}

print('\n' + '='*70)
print('EARLY WARNING ACTION PLAN')
print('='*70)

for risk_level, action in action_plan.items():
    count = (early_warnings['warning_level'] == risk_level).sum()
    print(f'\n{risk_level} ({count} loans):')
    print(f'  → {action}')

## Step 6: Save All Analysis for Report Generation

In [ ]:
# Save all analysis data for report generation
pricing_recs.to_csv('../outputs/metrics/pricing_recommendations.csv')

# Create summary dataframe for CRO report
report_summary = pd.DataFrame({
    'Current_Expected_Loss': [current_el],
    'Projected_Expected_Loss': [projected_el],
    'EL_Reduction': [el_reduction],
    'EL_Reduction_Pct': [el_reduction/current_el],
    'Additional_Income': [income_increase],
    'Total_Net_Benefit': [net_benefit],
})

report_summary.to_csv('../outputs/metrics/policy_impact_summary.csv', index=False)

print('\nAll analysis saved to outputs/')
print('Ready for CRO report generation!')